# Phase 6 validation-only pilot
Use two fresh sessions (ttm, moirai1). Target A100; actual GPU is recorded. Python 3.11/3.12. This is not a full experiment. No test targets/metrics. Run cells in order. Drive mounting requires YOUR authorization. Published phase-6 commit is required; do not pull during a run.

In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

assert (3, 11) <= sys.version_info[:2] <= (3, 12)
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
FAMILY = input("Model family (ttm or moirai1): ").strip()
assert FAMILY in ("ttm", "moirai1")
COMMIT = input("Full published phase-6 commit SHA: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)

In [ ]:
ROOT = Path("/content") / ("tsfm-pilot-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/tsfm-phase6")
OUT = PERSIST / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
CACHE = PERSIST / "public-source-cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("Persistent results and checkpoints:", OUT)
print("Stale running.lock: confirm old process is dead before manually removing only that lock.")

In [ ]:
ENV = Path("/content") / ("venv-pilot-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", f"python{version}-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", f"requirements/{FAMILY}-gpu.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
probe = (
    "import sys,torch; print(sys.executable,sys.version,torch.__version__,torch.version.cuda);"
    "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(),"
    "torch.cuda.get_device_properties(0).total_memory)"
)
subprocess.run([str(PY), "-c", probe], check=True)
print("All model processes use:", PY, "; kernel imports no vendor package")

In [ ]:
PINNED = ROOT / "results/manifests/pilot/prepared_data.json"
entries = json.loads(PINNED.read_text())
names = [n for n, e in entries.items() if e["status"] == "ready_with_warnings"]
verification = OUT / "download-verification.json"
if not verification.exists():
    subprocess.run(
        [
            str(PY),
            "-m",
            "tsfm_crossover.data.pilot_data",
            "--names",
            *names,
            "--cache",
            str(CACHE),
            "--verify-against",
            str(PINNED),
            "--output",
            str(verification),
        ],
        check=True,
    )
    verified = json.loads(verification.read_text())
    assert all(verified[n]["status"] == "ready_with_warnings" for n in names)
else:
    # On a fresh runtime, prepared files must still be regenerated from persistent raw.
    if any(not (ROOT / entries[n]["path"]).exists() for n in names):
        import uuid

        subprocess.run(
            [
                str(PY),
                "-m",
                "tsfm_crossover.data.pilot_data",
                "--names",
                *names,
                "--cache",
                str(CACHE),
                "--verify-against",
                str(PINNED),
                "--output",
                str(OUT / f"verification-{uuid.uuid4().hex}.json"),
            ],
            check=True,
        )
print("Prepared variants:", [(n, entries[n]["variant"]) for n in names])
print("Electricity is explicitly UCI 370-channel hourly, NOT the legacy 321-channel bundle.")

In [ ]:
BASE = [
    str(PY),
    "-m",
    "tsfm_crossover.experiments.pilot",
    "--output",
    str(OUT),
    "--expected-commit",
    COMMIT,
]
subprocess.run(BASE, check=True)
plan = json.loads((OUT / "plan.json").read_text())
jobs = [r for r in plan["conditions"] if r["family"] == FAMILY and r["status"] == "planned"]
for row in jobs:
    print(row["id"], row["kind"], row["dataset"], row["horizon"], row.get("learning_rate"))
print(
    "This session:",
    len(jobs),
    "groups. Feasibility: at most 14 batch attempts/group; stability: at most 200 steps/candidate.",
)
print("No automatic extra LR trials or transfer of trained weights between candidates.")

In [ ]:
assert input("Review plan above. Type RUN_A for bounded feasibility: ") == "RUN_A"
for row in sorted(
    [r for r in jobs if r["kind"] == "feasibility"], key=lambda r: (-r["horizon"], r["dataset"])
):
    done = subprocess.run([*BASE, "--condition-id", row["id"]], check=False)
    print(row["dataset"], row["horizon"], "exit:", done.returncode)
print("Inspect per-condition JSON. completed means ladder finished, not every batch passed.")

In [ ]:
assert input("Review feasibility results first. Type RUN_B for stability candidates: ") == "RUN_B"
for row in [r for r in jobs if r["kind"] == "stability"]:
    prerequisite = next(
        r
        for r in jobs
        if r["kind"] == "feasibility" and r["dataset"] == row["dataset"] and r["horizon"] == 96
    )
    evidence = OUT / prerequisite["id"] / "result.json"
    if not evidence.exists() or not json.loads(evidence.read_text()).get("fp32_batch1_supported"):
        print("blocked by feasibility:", row["dataset"], row["learning_rate"])
        continue
    done = subprocess.run([*BASE, "--condition-id", row["id"]], check=False)
    print(row["dataset"], row["learning_rate"], "exit:", done.returncode)
print("No convergence or final protocol decision is implied by reaching 200 steps.")

In [ ]:
from google.colab import files

archive = Path("/content") / f"{FAMILY}-phase6-pilot-results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and path.suffix in (".json", ".csv") and path.stat().st_size < 2000000:
            bundle.write(path, arcname=path.relative_to(OUT))
files.download(str(archive))
print(
    "Model checkpoints remain in your Drive; never upload them to Git. "
    "Return this small ZIP for review."
)